In [15]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import tool
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

from langgraph.graph import  StateGraph, START, END
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition

In [5]:
load_dotenv()

True

In [6]:
llm = ChatOllama(model="llama3.2")

In [12]:
import pypdf
loader = PyPDFLoader("intro-to-ml.pdf")
docs = loader.load()

In [13]:
len(docs)

392

In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
len(chunks)

973

In [16]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)
vector_store = Chroma(
    collection_name="my_documents", 
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

In [20]:
retriver = vector_store.as_retriever(search_type="similarity", search_kwargs={'k':4})

In [ ]:
@tool 
def rag_tool(query):
    """
    Retriver relavent information from the pdf document.
    Use this tool when the user asks factual /conceptual qquestions that mighrt be answered from the stored documents.
    """
    result = retriver.invoke(query)

    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return{
        'query': query, 
        'context': context, 
        'metadata': metadata
    }
